In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Dell\\OneDrive\\Desktop\\mlops-project\\chest-ct-ecs\\research'

In [3]:
os.chdir("../")

In [4]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int

In [5]:
# Simple imports
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

print("✅ Imports successful!")

✅ Imports successful!


In [6]:
# Read config
config = read_yaml(CONFIG_FILE_PATH)
params = read_yaml(PARAMS_FILE_PATH)

print("Config loaded!")
print(f"Config keys: {list(config.keys())}")
print(f"Params keys: {list(params.keys())}")

[2026-05-08 17:52:04,359: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-08 17:52:04,365: INFO: common: yaml file: params.yaml loaded successfully]
Config loaded!
Config keys: ['artifacts_root', 'data_ingestion', 'prepare_base_model']
Params keys: ['AUGMENTATION', 'IMAGE_SIZE', 'BATCH_SIZE', 'INCLUDE_TOP', 'EPOCHS', 'CLASSES', 'WEIGHTS', 'LEARNING_RATE']


In [7]:
# Get prepare_base_model config (simple dictionary access)
prepare_config = config.prepare_base_model

print("\nPrepare Base Model Config:")
print(f"  root_dir: {prepare_config.root_dir}")
print(f"  base_model_path: {prepare_config.base_model_path}")
print(f"  updated_base_model_path: {prepare_config.updated_base_model_path}")


Prepare Base Model Config:
  root_dir: artifacts/prepare_base_model
  base_model_path: artifacts/prepare_base_model/base_model.h5
  updated_base_model_path: artifacts/prepare_base_model/base_model_updated.h5


In [8]:
# Get parameters from params
print("\nParameters:")
print(f"  IMAGE_SIZE: {params.IMAGE_SIZE}")
print(f"  LEARNING_RATE: {params.LEARNING_RATE}")
print(f"  INCLUDE_TOP: {params.INCLUDE_TOP}")
print(f"  WEIGHTS: {params.WEIGHTS}")
print(f"  CLASSES: {params.CLASSES}")


Parameters:
  IMAGE_SIZE: [224, 224, 3]
  LEARNING_RATE: 0.0005
  INCLUDE_TOP: False
  WEIGHTS: imagenet
  CLASSES: 2


In [9]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
    
    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        
        create_directories([config.root_dir])
        
        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,           # From params.yaml
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
        )
        
        return prepare_base_model_config

In [10]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
from pathlib import Path
from cnnClassifier import logger

[2026-05-08 18:03:35,000: WARNING: module_wrapper: From c:\Users\Dell\anaconda3\envs\chest-ct-ecs\Lib\site-packages\keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.
]


In [11]:
# Cell 4: Define PrepareBaseModel class
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config
    
    def get_base_model(self):
        print("Getting base model...")
        
        self.model = tf.keras.applications.VGG16(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )
        
        self.save_model(path=self.config.base_model_path, model=self.model)
        print(f"Base model saved to: {self.config.base_model_path}")
    
    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        print("Preparing full model...")
        
        if freeze_all:
            for layer in model.layers:
                layer.trainable = False
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                layer.trainable = False
        
        flatten_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(
            units=classes,
            activation="softmax"
        )(flatten_in)
        
        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )
        
        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )
        
        full_model.summary()
        return full_model
    
    def update_base_model(self):
        print("Updating base model...")
        
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )
        
        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)
        print(f"Updated model saved to: {self.config.updated_base_model_path}")
    
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        model.save(path)
        print(f"Model saved: {path}")

In [12]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
    print("✅ Prepare Base Model completed successfully!")
except Exception as e:
    raise e

[2026-05-08 18:12:19,587: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-08 18:12:19,592: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-08 18:12:19,595: INFO: common: created directory at: artifacts]
[2026-05-08 18:12:19,597: INFO: common: created directory at: artifacts/prepare_base_model]
Getting base model...
[2026-05-08 18:12:20,129: WARNING: module_wrapper: From c:\Users\Dell\anaconda3\envs\chest-ct-ecs\Lib\site-packages\keras\src\backend.py:1398: The name tf.executing_eagerly_outside_functions is deprecated. Please use tf.compat.v1.executing_eagerly_outside_functions instead.
]
[2026-05-08 18:12:20,429: WARNING: module_wrapper: From c:\Users\Dell\anaconda3\envs\chest-ct-ecs\Lib\site-packages\keras\src\layers\pooling\max_pooling2d.py:161: The name tf.nn.max_pool is deprecated. Please use tf.nn.max_pool2d instead.
]
[2026-05-08 18:12:20,993: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be buil

c:\Users\Dell\anaconda3\envs\chest-ct-ecs\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0         
          